# yt-clips Kaggle Worker (T4-Optimized)
## Settings → Accelerator → GPU × 2 (T4)
Run all cells. Worker listens for jobs from your Mac.
Add host reference photos to the `photos/` folder for automatic face detection & tracking.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

WORKDIR = Path("/kaggle/working")
REPO_URL = "https://github.com/fearless16/yt-clips.git"
REPO_MARKER = Path("pipeline.py")
REPO_STAGING_DIR = Path("_repo")

def run(cmd: str, *, check: bool = False, capture: bool = False):
    return subprocess.run(
        cmd,
        shell=True,
        check=check,
        capture_output=capture,
        text=True,
    )

def print_gpu_status() -> None:
    gpu = run(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null",
        capture=True,
    ).stdout.strip()
    print(f"GPU: {gpu or 'NONE! Enable GPU × 2 in Settings → Accelerator'}")

def bootstrap_repo() -> None:
    os.chdir(WORKDIR)
    print(f"Working dir: {os.getcwd()}")

    if REPO_MARKER.exists():
        print("Repo already present.")
        return

    print("Cloning yt-clips repo...")
    run(f"git clone --depth 1 {REPO_URL} {REPO_STAGING_DIR}", check=True)

    for item in REPO_STAGING_DIR.iterdir():
        dest = Path(item.name)
        if dest.exists():
            if dest.is_dir():
                shutil.rmtree(dest)
            else:
                dest.unlink()
        shutil.move(str(item), ".")

    shutil.rmtree(REPO_STAGING_DIR)
    print("Repo ready.")

print_gpu_status()
bootstrap_repo()
print(f"CWD: {os.getcwd()}")

In [ ]:
def install_system_deps() -> None:
    print("Installing system deps...")
    steps = [
        ("apt update", "apt-get update -qq > /dev/null 2>&1"),
        ("aria2 + ffmpeg", "apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1"),
        ("localtunnel", "npm install -g localtunnel > /dev/null 2>&1"),
    ]
    for desc, cmd in steps:
        print(f"  -> {desc}...")
        result = run(cmd, capture=True)
        if result.returncode != 0 and result.stderr:
            print(f"    ⚠ stderr: {result.stderr.strip()[:200]}")
    print("  ✅ System deps installed")

install_system_deps()

In [ ]:
def install_python_deps() -> None:
    print("Installing T4-optimized deps...")

    # ═══ ROOT CAUSE: basicsr==1.4.2 (PyPI, Aug 2022) imports ═══
    # PIL._typing._Ink which was REMOVED in Pillow 10.0 (July 2023).
    # ALL PyPI versions of basicsr/realesrgan/gfpgan are stale 2022
    # packages — must pin Pillow BEFORE installing any of them.
    print("  -> [1/7] Pinning Pillow==9.5.0 (basicsr compat)...")
    r = run("pip install -q --force-reinstall --no-deps 'Pillow==9.5.0'", capture=True)
    if r.returncode != 0:
        print(f"    ⚠ Pillow 9.5.0 failed, trying <10: {r.stderr.strip()[:200]}")
        run("pip install -q --force-reinstall --no-deps 'Pillow<10'", capture=True)

    # ═══ Core deps ═══
    core_deps = (
        "yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
        "filterpy scipy google-genai google-generativeai openai python-dotenv "
        "ultralytics face_recognition"
    )
    print("  -> [2/7] Core deps...")
    r = run(f"pip install -q {core_deps}", capture=True)
    if r.returncode != 0:
        print(f"    ⚠ Core deps errors: {r.stderr.strip()[:200]}")

    # Re-pin Pillow after core deps (some may have upgraded it)
    run("pip install -q --force-reinstall --no-deps 'Pillow==9.5.0'", capture=True)

    # ═══ Build deps ═══
    print("  -> [3/7] Cython + numpy (build deps)...")
    run("pip install -q cython numpy", capture=True)

    # ═══ basicsr: install from GitHub master (has May 2024 ═══
    # torchvision fix: functional_tensor → functional)
    print("  -> [4/7] basicsr (GitHub master for torchvision compat)...")
    r = run(
        "pip install -q --no-deps 'git+https://github.com/XPixelGroup/BasicSR.git'",
        capture=True,
    )
    if r.returncode != 0:
        print(f"    ⚠ basicsr GitHub failed, falling back to PyPI: {r.stderr.strip()[:200]}")
        r = run("pip install -q --no-deps basicsr", capture=True)
        if r.returncode != 0:
            print(f"    ❌ basicsr install failed: {r.stderr.strip()[:300]}")

    # Install basicsr runtime deps (but NOT Pillow — already pinned)
    print("  -> [5/7] basicsr runtime deps...")
    run(
        "pip install -q addict future lmdb pyyaml requests scikit-image scipy yapf",
        capture=True,
    )

    # Re-pin Pillow again after each stage
    run("pip install -q --force-reinstall --no-deps 'Pillow==9.5.0'", capture=True)

    # ═══ Real-ESRGAN + GFPGAN + facexlib ═══
    print("  -> [6/7] Real-ESRGAN + GFPGAN + FaceXLib...")
    r = run(
        "pip install -q --no-deps realesrgan gfpgan facexlib",
        capture=True,
    )
    if r.returncode != 0:
        print(f"    ⚠ Face model errors: {r.stderr.strip()[:300]}")

    # ═══ Final Pillow lock + verify ═══
    # Force-reinstall with --no-deps so dependency resolution can't
    # upgrade Pillow back to 10+ (ultralytics etc. require Pillow>=10)
    run("pip install -q --force-reinstall --no-deps 'Pillow==9.5.0'", capture=True)

    print("\n  [7/7] Verifying imports...")
    ok = True
    for mod in ["realesrgan", "gfpgan", "face_recognition"]:
        try:
            __import__(mod)
            print(f"    ✅ {mod}")
        except Exception as e:
            print(f"    ❌ {mod}: {e}")
            ok = False
    if not ok:
        print("    ⚠ Some GPU models missing — super-res may not work")

    try:
        import PIL
        print(f"    Pillow version: {PIL.__version__}")
    except Exception:
        pass

    print("\n  ✅ T4-optimized deps installed")

install_python_deps()

In [ ]:
import importlib
import torch

def verify_environment() -> None:
    print("=" * 55)
    print("  GPU STATUS")
    print("=" * 55)
    print(f"  CUDA available: {torch.cuda.is_available()}")
    print(f"  GPU count: {torch.cuda.device_count()}")

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        gb = props.total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({gb:.1f} GB)")

    print("\n" + "=" * 55)
    print("  MODEL STATUS")
    print("=" * 55)

    modules = [
        "realesrgan",
        "gfpgan",
        "face_recognition",
        "faster_whisper",
    ]

    for name in modules:
        try:
            importlib.import_module(name)
            print(f"  ✅ {name}")
        except Exception:
            print(f"  ❌ {name}")

    print("\n  Ready for AI pipeline!")

verify_environment()

In [ ]:
import os
from pathlib import Path

def download_models() -> None:
    """Download GFPGAN + Real-ESRGAN model weights."""
    model_dir = Path("experiments/pretrained_models")
    model_dir.mkdir(parents=True, exist_ok=True)

    models = {
        "GFPGANv1.4.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
        "RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    }

    for name, url in models.items():
        target = model_dir / name
        if target.exists():
            print(f"  ✅ {name} already exists")
            continue
        print(f"  -> Downloading {name}...")
        r = run(f"wget -q -O {target} {url}", capture=True)
        if r.returncode != 0 or not target.exists():
            print(f"    ❌ Failed to download {name}")
            if target.exists():
                target.unlink()
        else:
            print(f"    ✅ {name} downloaded ({target.stat().st_size / 1e6:.1f} MB)")

    # Also copy to weights/ for super_res.py
    weights_dir = Path("weights")
    weights_dir.mkdir(exist_ok=True)
    for name in models:
        src = model_dir / name
        dst = weights_dir / name
        if src.exists() and not dst.exists():
            dst.symlink_to(src.resolve())

download_models()

In [ ]:
def setup_runtime() -> None:
    try:
        import utils.torchvision_compat  # noqa: F401
        print("  ✅ torchvision compat shim applied")
    except Exception as e:
        print(f"  ⚠ torchvision compat shim skipped: {e}")

    def clear_vram():
        """Free GPU memory between pipeline stages."""
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    globals()["clear_vram"] = clear_vram

    folders = [
        "input",
        "temp",
        "transcripts",
        "highlights",
        "shorts",
        "logs",
        "photos",
    ]
    for folder in folders:
        Path(folder).mkdir(exist_ok=True)

    print("  ✅ VRAM cleanup helper ready")

setup_runtime()

In [ ]:
import time

def kill_old_processes() -> None:
    run("pkill -f 'python watcher.py' 2>/dev/null || true")
    run("pkill -f 'lt --port' 2>/dev/null || true")
    time.sleep(1)

def start_services() -> None:
    kill_old_processes()

    with open("watcher.log", "w") as watcher_log, \
         open("tunnel.log", "w") as tunnel_log:

        subprocess.Popen(
            [sys.executable, "watcher.py"],
            stdout=watcher_log,
            stderr=subprocess.STDOUT,
        )
        time.sleep(2)

        subprocess.Popen(
            ["lt", "--port", "5000"],
            stdout=tunnel_log,
            stderr=subprocess.STDOUT,
        )
        time.sleep(5)

    with open("tunnel.log") as f:
        for line in f:
            line = line.strip()
            if "://" in line:
                Path("kaggle_url.txt").write_text(line)
                print(f"Tunnel URL: {line}")
                break

start_services()

In [ ]:
print("=" * 55)
print("  WORKER IS ONLINE!")
print("=" * 55)
print("On Mac, run:")
print('  ./automate.sh "URL"')
print("  -> Option 2 (Remote Run)")

try:
    last_pos = 0
    last_inode = None

    while True:
        time.sleep(10)

        try:
            st = Path("watcher.log").stat()
        except FileNotFoundError:
            continue

        if last_inode is not None and st.st_ino != last_inode:
            last_pos = 0

        last_inode = st.st_ino

        with open("watcher.log", "r") as f:
            f.seek(last_pos)
            for line in f.readlines():
                line = line.strip()
                if line:
                    print(line)
            last_pos = f.tell()

except KeyboardInterrupt:
    print("Stopped.")